In [ ]:
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
import scipy.sparse as sp
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import harmonypy as hm
import scanpy.external as sce
import squidpy as sq
import spatialdata as sd
import spatialdata_plot as sdp
from spatialdata.models.models import ShapesModel
import os
from matplotlib.colors import TwoSlopeNorm, Normalize
from matplotlib.backends.backend_pdf import PdfPages
from scipy.spatial import ConvexHull
from shapely.geometry import Polygon

# QC

In [ ]:
tma6_og = sc.read("/homevol/kk/analysis/OVA_TMA/TMA6_notebook/adjustments_mis_assigned_transcripts/xenium_mistic/tma6_xenium_clean_mistic.h5ad")

In [ ]:
tma6_og

In [ ]:
## add the meta info back
tma6l_meta = pd.read_csv('/homevol/kk/analysis/OVA_TMA/Old/TMA5_TMA6/tma6l_og_metadata.csv')
tma6r_meta = pd.read_csv('/homevol/kk/analysis/OVA_TMA/Old/TMA5_TMA6/tma6r_og_metadata.csv')
tma6_meta = pd.concat([tma6l_meta, tma6r_meta], axis=0, ignore_index=True)
tma6_meta = tma6_meta.set_index('cell_id')


In [ ]:
# Ensure ordering matches
tma6_meta = tma6_meta.loc[tma6_og.obs.index]

# # Overwrite obs
tma6_og.obs[['arrayID', 'tma', 'PatientID', 'Diagnosis']] = tma6_meta[['arrayID', 'tma', 'PatientID', 'Diagnosis']]


In [ ]:
## drop out unassigned cells or TMAs

keep = tma6_og.obs["arrayID"] != "unassigned"

if "PatientID" in tma6_og.obs.columns:
    keep &= tma6_og.obs["PatientID"].notna()

if "Diagnosis" in tma6_og.obs.columns:
    keep &= tma6_og.obs["Diagnosis"].notna()

tma6_og = tma6_og[keep].copy()


In [ ]:
sc.pp.calculate_qc_metrics(tma6_og, inplace=True, log1p=True, percent_top=None)


In [ ]:
tma6_og

In [ ]:
## Filter out low quality cells
tma6 = tma6_og[
    (tma6_og.obs["total_counts"] >= 10) &
    (tma6_og.obs["n_genes_by_counts"] > 0)
].copy()

In [ ]:
tma6

In [ ]:
## check TMA core wise cell count distribution

# Count cells per array
array_counts = (
    tma6.obs.groupby("arrayID")
    .size()
    .reset_index(name="cell_count")
)
array_meta = (
    tma6.obs.groupby("arrayID")[["Diagnosis", "PatientID"]]
    .first()
    .reset_index()
)

array_counts = (
    array_counts
    .merge(array_meta, on="arrayID")
    .sort_values("cell_count", ascending=True)
)

In [ ]:
## Filter out TMA cores with less than 100 cells in total

valid_arrays = array_counts.loc[array_counts["cell_count"] >= 100, "arrayID"]

# Subset AnnData to only those arrays
tma6 = tma6[tma6.obs["arrayID"].isin(valid_arrays)].copy()

tma6.obs.head()

In [ ]:
tma6_og.obs['PatientID'].nunique()

In [ ]:
tma6

In [ ]:
tma6.X[1:5, 1:5].toarray()

In [ ]:
tma6.layers["raw_counts"] = tma6.X.copy()

In [ ]:
sc.pp.normalize_total(tma6)
sc.pp.log1p(tma6)
sc.tl.pca(tma6)
sc.pp.neighbors(tma6, n_pcs= 10)
sc.tl.umap(tma6)
sc.tl.leiden(tma6, key_added="leiden_res_0.1", resolution=0.1)


In [ ]:
sc.pl.umap(tma6, color="leiden_res_0.1", 
            legend_loc="on data",
            legend_fontsize=18,
            legend_fontoutline=2
           )

In [ ]:
sc.tl.rank_genes_groups(tma6, groupby= "leiden_res_0.1", method='wilcoxon', key_added="rank_genes_0.1")

In [ ]:
sc.pl.rank_genes_groups_dotplot(tma6, 
                                groupby="leiden_res_0.1", 
                                key="rank_genes_0.1",
                                n_genes=5)

In [ ]:
sc.pl.rank_genes_groups_dotplot(tma6, 
                                groupby="leiden_res_0.1", 
                                key="rank_genes_0.1",
                                n_genes=5,
                                values_to_plot="logfoldchanges",
                                cmap="bwr",
                                vcenter=0,
                                vmin=-5,
                                vmax=5,
                                min_logfoldchange=2)

## Supervised annotation with Decoupler (ULM)

In [ ]:
markers = pd.read_csv(
    "/homevol/kk/analysis/OVA_TMA/TMA6_notebook/adjustments_mis_assigned_transcripts/supervised_annotation/v2/panel_annot_broad_labels_for_annot_tools_AF_v2.csv",
    encoding="cp1252"
)

In [ ]:
## marker list prep

markers = markers.drop_duplicates()

markers = markers.rename(columns={"Adjusted_group": "source", "gene": "target"})

markers = markers.loc[:, ~markers.columns.duplicated()]

markers = markers[["source", "target"]]

In [ ]:
dc.run_ulm(
        mat=tma6,
        net=markers,
        weight = None, 
        min_n= 1,
        verbose= True,
        use_raw= False
    )

# assign top-scoring cell type
tma6.obs["decoupler"] = tma6.obsm["ulm_estimate"].idxmax(axis=1)


In [ ]:
tma6

In [ ]:
### Filter mismatched cells between coarse leiden and decoupler

dec = tma6.obs["decoupler"].astype(str)
cl  = tma6.obs["leiden_res_0.1"].astype(str)

malignant_only = {"Tumour"}
mismatch = np.full(dec.shape, False)


mismatch_cl0 = (cl == "0") & (~dec.isin(["Tumour", "Proliferating"]))

mismatch_cl12 = cl.isin(["1", "2"]) & (dec.isin(malignant_only))

mismatch = mismatch_cl0 | mismatch_cl12

tma6.obs["malignancy_mismatch"] = np.where(mismatch, "Mismatch", "Match")


In [ ]:
tma6.obs["malignancy_mismatch"].value_counts()

In [ ]:
pd.crosstab(
    tma6.obs["decoupler"],
    tma6.obs["malignancy_mismatch"]
)


In [ ]:
## filter out mismatched cells

tma6_filt = tma6[tma6.obs["malignancy_mismatch"] != "Mismatch"].copy()

In [ ]:
tma6_filt

In [ ]:
tma6_filt.write("/homevol/kk/analysis/OVA_TMA/TMA6_notebook/adjustments_mis_assigned_transcripts/supervised_annotation/v3/tma6_xen_mist_coarse_decoupler_filt.h5ad")

# sublustering of the coarse decoupler lineage types into specific sub-populations

In [ ]:
tma6 = sc.read("/homevol/kk/analysis/OVA_TMA/TMA6_notebook/adjustments_mis_assigned_transcripts/supervised_annotation/v3/tma6_xen_mist_coarse_decoupler_filt.h5ad")
tma6

In [ ]:
# Get all fine labels
celltypes = (
    markers["fine_label_AF"]
    .dropna()
    .unique()
    .tolist()
)

celltypes_ordered = [ct for ct in celltypes if ct != "-"]
if "-" in celltypes:
    celltypes_ordered.append("-")

celltype_markers = {
    ct: sorted(
        markers.loc[markers["fine_label_AF"] == ct, "gene"]
        .unique()
        .tolist()
    )
    for ct in celltypes_ordered
}


## T-cells

In [ ]:
tma6_t = tma6[tma6.obs["decoupler"] == "T cells"].copy()
tma6_t

In [ ]:
tma6_t.X = tma6_t.layers['raw_counts'].copy()

In [ ]:
tma6_t.X[1:5, 1:5].toarray()

In [ ]:
sc.pp.normalize_total(tma6_t)
sc.pp.log1p(tma6_t)
sc.tl.pca(tma6_t)

In [ ]:
sc.pl.pca_variance_ratio(tma6_t, log=True, n_pcs=50)


In [ ]:
sc.pp.neighbors(tma6_t)
sc.tl.umap(tma6_t)
sc.tl.leiden(tma6_t)

In [ ]:
tma6_t

In [ ]:
markers_t = markers[markers["Adjusted_group"] == "T cells"].copy()
markers_t = markers_t[markers_t["fine_label_AF"] != "-"]
markers_t = markers_t.drop_duplicates()
markers_t = markers_t.rename(columns={"fine_label_AF": "source", "gene": "target"})
markers_t = markers_t.loc[:, ~markers_t.columns.duplicated()]
markers_t = markers_t[["source", "target"]]
markers_t

In [ ]:
dc.run_ulm(
        mat=tma6_t,
        net=markers_t,
        weight = None, 
        min_n= 1,
        verbose= True,
        use_raw= False
    )

# assign top-scoring cell type
tma6_t.obs["decoupler_t"] = tma6_t.obsm["ulm_estimate"].idxmax(axis=1)


## B-cells

In [ ]:
tma6_b = tma6[tma6.obs["decoupler"] == "B_plasma cells"].copy()
tma6_b

In [ ]:
tma6_b.X = tma6_b.layers['raw_counts'].copy()

In [ ]:
tma6_b.X[1:5, 1:5].toarray()

In [ ]:
sc.pp.normalize_total(tma6_b)
sc.pp.log1p(tma6_b)
sc.tl.pca(tma6_b)

In [ ]:
sc.pl.pca_variance_ratio(tma6_b, log=True, n_pcs=50)


In [ ]:
sc.pp.neighbors(tma6_b)
sc.tl.umap(tma6_b)
sc.tl.leiden(tma6_b)

In [ ]:
markers_b = markers[markers["Adjusted_group"] == "B_plasma cells"].copy()
markers_b = markers_b[markers_b["fine_label_AF"] != "-"]
markers_b = markers_b.drop_duplicates()
markers_b = markers_b.rename(columns={"fine_label_AF": "source", "gene": "target"})
markers_b = markers_b.loc[:, ~markers_b.columns.duplicated()]
markers_b = markers_b[["source", "target"]]
markers_b

In [ ]:
dc.run_ulm(
        mat=tma6_b,
        net=markers_b,
        weight = None, 
        min_n= 1,
        verbose= True,
        use_raw= False
    )

# assign top-scoring cell type
tma6_b.obs["decoupler_b"] = tma6_b.obsm["ulm_estimate"].idxmax(axis=1)


## Mono_macrophages

In [ ]:
tma6_mac = tma6[tma6.obs["decoupler"] == "Monocyte_Macrophages"].copy()
tma6_mac

In [ ]:
tma6_mac.X = tma6_mac.layers['raw_counts'].copy()

In [ ]:
tma6_mac.X[1:5, 1:5].toarray()

In [ ]:
sc.pp.normalize_total(tma6_mac)
sc.pp.log1p(tma6_mac)
sc.tl.pca(tma6_mac)

In [ ]:
# Inspect variance explained
sc.pl.pca_variance_ratio(tma6_mac, log=True, n_pcs=50)


In [ ]:
sc.pp.neighbors(tma6_mac)
sc.tl.umap(tma6_mac)
sc.tl.leiden(tma6_mac)

In [ ]:
tma6_mac

In [ ]:
markers_mac = markers[markers["Adjusted_group"] == "Monocyte_Macrophages"].copy()
markers_mac = markers_mac[markers_mac["fine_label_AF"] != "-"]
markers_mac = markers_mac.drop_duplicates()
markers_mac = markers_mac.rename(columns={"fine_label_AF": "source", "gene": "target"})
markers_mac = markers_mac.loc[:, ~markers_mac.columns.duplicated()]
markers_mac = markers_mac[["source", "target"]]
markers_mac

In [ ]:
dc.run_ulm(
        mat=tma6_mac,
        net=markers_mac,
        weight = None, 
        min_n= 1,
        verbose= True,
        use_raw= False
    )

# assign top-scoring cell type
tma6_mac.obs["decoupler_mac"] = tma6_mac.obsm["ulm_estimate"].idxmax(axis=1)


## Fibroblasts

In [ ]:
tma6_fib = tma6[tma6.obs["decoupler"] == "Fibroblast"].copy()
tma6_fib

In [ ]:
tma6_fib.X = tma6_fib.layers['raw_counts'].copy()

In [ ]:
tma6_fib.X[1:5, 1:5].toarray()

In [ ]:
sc.pp.normalize_total(tma6_fib)
sc.pp.log1p(tma6_fib)
sc.tl.pca(tma6_fib)

In [ ]:
# Inspect variance explained
sc.pl.pca_variance_ratio(tma6_fib, log=True, n_pcs=50)


In [ ]:
sc.pp.neighbors(tma6_fib)
sc.tl.umap(tma6_fib)
sc.tl.leiden(tma6_fib)

In [ ]:
tma6_fib

In [ ]:
markers_fib = markers[markers["Adjusted_group"] == "Fibroblast"].copy()
markers_fib = markers_fib[markers_fib["fine_label_AF"] != "-"]
markers_fib = markers_fib.drop_duplicates()
markers_fib = markers_fib.rename(columns={"fine_label_AF": "source", "gene": "target"})
markers_fib = markers_fib.loc[:, ~markers_fib.columns.duplicated()]
markers_fib = markers_fib[["source", "target"]]
markers_fib

In [ ]:
dc.run_ulm(
        mat=tma6_fib,
        net=markers_fib,
        weight = None, 
        min_n= 1,
        verbose= True,
        use_raw= False
    )

# assign top-scoring cell type
tma6_fib.obs["decoupler_fib"] = tma6_fib.obsm["ulm_estimate"].idxmax(axis=1)


## Endothelial

In [ ]:
sc.set_figure_params(fontsize=14) 
plt.rcParams['patch.edgecolor'] = 'black'
sc.set_figure_params(figsize=(8, 6)) 

In [ ]:
tma6_endo = tma6[tma6.obs["decoupler"] == "Endothelial"].copy()
tma6_endo

In [ ]:
tma6_endo.X = tma6_endo.layers['raw_counts'].copy()

In [ ]:
tma6_endo.X[1:5, 1:5].toarray()

In [ ]:
sc.pp.normalize_total(tma6_endo)
sc.pp.log1p(tma6_endo)
sc.tl.pca(tma6_endo)

In [ ]:
# Inspect variance explained
sc.pl.pca_variance_ratio(tma6_endo, log=True, n_pcs=50)


In [ ]:
sc.pp.neighbors(tma6_endo)
sc.tl.umap(tma6_endo)
sc.tl.leiden(tma6_endo)

In [ ]:
tma6_endo

In [ ]:
markers_endo = markers[markers["Adjusted_group"] == "Endothelial"].copy()
markers_endo = markers_endo[markers_endo["fine_label_AF"] != "-"]
markers_endo = markers_endo.drop_duplicates()
markers_endo = markers_endo.rename(columns={"fine_label_AF": "source", "gene": "target"})
markers_endo = markers_endo.loc[:, ~markers_endo.columns.duplicated()]
markers_endo = markers_endo[["source", "target"]]
markers_endo

In [ ]:
dc.run_ulm(
        mat=tma6_endo,
        net=markers_endo,
        weight = None, 
        min_n= 1,
        verbose= True,
        use_raw= False
    )

# assign top-scoring cell type
tma6_endo.obs["decoupler_endo"] = tma6_endo.obsm["ulm_estimate"].idxmax(axis=1)


## incorporating the decoupler subtypes to the main anndata

In [ ]:
fine_tables = [
    tma6_t.obs[["cell_id", "decoupler_t"]].rename(columns={"decoupler_t": "decoupler_fine"}),
    tma6_fib.obs[["cell_id", "decoupler_fib"]].rename(columns={"decoupler_fib": "decoupler_fine"}),
    tma6_endo.obs[["cell_id", "decoupler_endo"]].rename(columns={"decoupler_endo": "decoupler_fine"}),
    tma6_mac.obs[["cell_id", "decoupler_mac"]].rename(columns={"decoupler_mac": "decoupler_fine"}),
    tma6_b.obs[["cell_id", "decoupler_b"]].rename(columns={"decoupler_b": "decoupler_fine"}),
]

fine_df = pd.concat(fine_tables, axis=0)
fine_df = fine_df.set_index("cell_id")
fine_df.head()

In [ ]:
tma6.obs["decoupler_fine"] = tma6.obs["decoupler"].astype("category")

new_cats = pd.Index(fine_df["decoupler_fine"].unique())
tma6.obs["decoupler_fine"] = tma6.obs["decoupler_fine"].cat.add_categories(new_cats)

tma6.obs.loc[fine_df.index, "decoupler_fine"] = fine_df["decoupler_fine"]
tma6.obs["decoupler_fine"] = tma6.obs["decoupler_fine"].cat.remove_unused_categories()

In [ ]:
tma6.write_h5ad("/homevol/kk/analysis/OVA_TMA/TMA6_notebook/adjustments_mis_assigned_transcripts/supervised_annotation/v3/tma6_xen_mist_coarse_decoupler_filt_fine.h5ad")

In [ ]:
os.chdir('/homevol/kk/analysis/OVA_TMA/TMA6_notebook/adjustments_mis_assigned_transcripts/supervised_annotation/v3/survival')

In [ ]:
tma6 = sc.read("/homevol/kk/analysis/OVA_TMA/TMA6_notebook/adjustments_mis_assigned_transcripts/supervised_annotation/v3/tma6_xen_mist_coarse_decoupler_filt_fine.h5ad")

In [ ]:
## sanity checks first

immune_labels = [
    "B_plasma cells", "Monocyte_Macrophages", "T cells", "Dendritic cells",
    "Mast cells", "Neutrophils", "NK cells", "pDC"
]

stromal_labels = [
    "Endothelial", "Fibroblast", "Adipocytes", "Mesothelial cells",
    "Myofibroblasts", "Pericyte", "Schwann cells"
]

tumour_labels = ["Tumour", "Proliferating"]


In [ ]:
tma6_tumour = tma6[tma6.obs["Diagnosis"] == "Tumour"].copy()


In [ ]:
def map_coarsetype(x):
    if x in tumour_labels:
        return "Tumour"
    elif x in immune_labels:
        return "Immune"
    elif x in stromal_labels:
        return "Stromal"
    else:
        return "Other"

tma6_tumour.obs["decoupler_coarse"] = tma6_tumour.obs["decoupler"].map(map_coarsetype)


In [ ]:
sc.pl.umap(tma6_tumour, color = "decoupler_coarse")

In [ ]:
patient_counts = (
    tma6_tumour.obs
       .groupby(["PatientID", "decoupler_coarse"])
       .size()
       .unstack(fill_value=0)
)

for col in ["Tumour", "Immune", "Stromal", "Other"]:
    if col not in patient_counts.columns:
        patient_counts[col] = 0

patient_counts = patient_counts.sort_index()

patient_counts.to_csv("tma6_tumour_decoupler_coarse_label_counts.csv")

patient_counts.head()


In [ ]:
patient_frac = patient_counts.div(patient_counts.sum(axis=1), axis=0)
patient_frac.head()


In [ ]:
patient_frac_sorted = (
    patient_frac
    .sort_values(by=["Tumour", "Stromal"], ascending=[False, False])
)

sns.set_style("white")



with plt.rc_context({"font.size": 16}):
    
    ax = patient_frac_sorted[["Tumour", "Immune", "Stromal"]].plot(
        kind="bar",
        stacked=True,
        figsize=(21, 7)
    )
    
    ax.set_ylabel("Fraction of cells", fontsize=16)
    ax.set_xlabel("PatientID", fontsize=16)
    ax.tick_params(axis="x", labelsize=16)
    ax.tick_params(axis="y", labelsize=16)
    
    ax.legend(
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        borderaxespad=0,
        fontsize = 16
    )
    
    plt.tight_layout()
    plt.show()



In [ ]:
patient_counts = (
    tma6_tumour.obs
       .groupby(["PatientID", "decoupler"])
       .size()
       .unstack(fill_value=0)
)

patient_counts = patient_counts.sort_index()

patient_counts.to_csv("tma6_tumour_decoupler_broad_label_counts.csv")

patient_counts.head()


In [ ]:
patient_frac = patient_counts.div(patient_counts.sum(axis=1), axis=0)
patient_frac.head()


In [ ]:
patient_frac_sorted = (
    patient_frac
    .sort_values(by=["Tumour", "Fibroblast"], ascending=[False, False])
)

sns.set_style("white")

with plt.rc_context({"font.size": 16}):

    fig, ax = plt.subplots(figsize=(21, 7))

    patient_frac_sorted.plot(
        kind="bar",
        stacked=True,
        ax=ax,
        colormap="tab20"
    )

    ax.set_ylabel("Fraction of cells", fontsize=16)
    ax.set_xlabel("PatientID", fontsize=16)

    ax.tick_params(axis="x", labelsize=16, rotation=90)
    ax.tick_params(axis="y", labelsize=16)

    ax.legend(
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        borderaxespad=0,
        fontsize=16
    )

    plt.tight_layout()
    plt.show()


## calculation of celltype fractions/abundance

In [ ]:
tma6_tumour

In [ ]:
### calculation of fraction
# For each patient -
# fraction of each broad_label ("decoupler") across it's coarse_label ("decoupler_coarse")
# fraction of each fine_label (where fine_label is not same as broad_label, "decoupler_fine") across it's associated broad_label ("decoupler")
# fraction of each fine_label (where fine_label is not same as broad_label, "decoupler_fine") across it's associated coarse_label ("decoupler_coarse")

In [ ]:
df = tma6_tumour.obs[[
    "PatientID", "decoupler", "decoupler_fine", "decoupler_coarse"
]].copy()

for col in ["PatientID", "decoupler", "decoupler_fine", "decoupler_coarse"]:
    df[col] = df[col].astype("category")

all_patients = df["PatientID"].cat.categories
all_broad   = df["decoupler"].cat.categories
all_fine    = df["decoupler_fine"].cat.categories
all_coarse  = df["decoupler_coarse"].cat.categories


In [ ]:
bwc_counts = (
    df.groupby(["PatientID", "decoupler_coarse", "decoupler"])
      .size()
      .rename("n")
      .reset_index()
)

bwc_denom = (
    df.groupby(["PatientID", "decoupler_coarse"])
      .size()
      .rename("denom")
      .reset_index()
)

broad_within_coarse_long = (
    bwc_counts
    .merge(bwc_denom, on=["PatientID", "decoupler_coarse"], how="left")
)

broad_within_coarse_long["fraction"] = (
    broad_within_coarse_long["n"] / broad_within_coarse_long["denom"]
)

broad_to_coarse = df[["decoupler", "decoupler_coarse"]].drop_duplicates()

valid_broad_combos = (
    pd.MultiIndex.from_product(
        [all_patients, all_broad],
        names=["PatientID", "decoupler"]
    )
    .to_frame(index=False)
    .merge(broad_to_coarse, on="decoupler", how="left")
)

broad_within_coarse_long = (
    valid_broad_combos
    .merge(
        broad_within_coarse_long,
        on=["PatientID", "decoupler", "decoupler_coarse"],
        how="left"
    )
    .fillna({"n": 0, "fraction": 0})
)

broad_within_coarse_long.to_csv("tma6_tumour_decoupler_frac_broad_on_coarse.csv", index = False)


In [ ]:
fine_df = df[df["decoupler_fine"].astype(str) != df["decoupler"].astype(str)]

broad_with_fine = fine_df["decoupler"].unique()


fine_to_broad = (
    df[["decoupler_fine", "decoupler"]]
    .drop_duplicates()
    .query("decoupler in @broad_with_fine")
)

valid_fine_broad = (
    pd.MultiIndex.from_product(
        [all_patients, fine_to_broad["decoupler_fine"].unique()],
        names=["PatientID", "decoupler_fine"]
    )
    .to_frame(index=False)
    .merge(fine_to_broad, on="decoupler_fine", how="left")
)

fwb_counts = (
    fine_df.groupby(["PatientID", "decoupler", "decoupler_fine"])
           .size().rename("n").reset_index()
)

fwb_denom = (
    df.groupby(["PatientID", "decoupler"])
      .size().rename("denom").reset_index()
)

fine_within_broad_long = (
    fwb_counts
    .merge(fwb_denom, on=["PatientID", "decoupler"], how="left")
)
fine_within_broad_long["fraction"] = (
    fine_within_broad_long["n"] / fine_within_broad_long["denom"]
)

fine_within_broad_long = (
    valid_fine_broad
    .merge(
        fine_within_broad_long,
        on=["PatientID", "decoupler", "decoupler_fine"],
        how="left"
    )
    .fillna({"n": 0, "fraction": 0})
)

fine_within_broad_long.to_csv("tma6_tumour_decoupler_frac_fine_on_broad.csv", index = False)

In [ ]:
fine_df = df[df["decoupler_fine"].astype(str) != df["decoupler"].astype(str)]

broad_with_fine = fine_df["decoupler"].unique()

fine_to_coarse = (
    df[["decoupler", "decoupler_fine", "decoupler_coarse"]]
    .drop_duplicates()
    .query("decoupler in @broad_with_fine")
)

valid_fine_coarse = (
    pd.MultiIndex.from_product(
        [all_patients, fine_to_coarse["decoupler_fine"].unique()],
        names=["PatientID", "decoupler_fine"]
    )
    .to_frame(index=False)
    .merge(fine_to_coarse, on="decoupler_fine", how="left")
)

fwc_counts = (
    fine_df.groupby(["PatientID", "decoupler_coarse", "decoupler_fine"])
           .size().rename("n").reset_index()
)

fwc_denom = (
    df.groupby(["PatientID", "decoupler_coarse"])
      .size().rename("denom").reset_index()
)

fine_within_coarse_long = (
    fwc_counts
    .merge(fwc_denom, on=["PatientID", "decoupler_coarse"], how="left")
)

fine_within_coarse_long["fraction"] = (
    fine_within_coarse_long["n"] / fine_within_coarse_long["denom"]
)

fine_within_coarse_long = (
    valid_fine_coarse
    .merge(
        fine_within_coarse_long,
        on=["PatientID", "decoupler_coarse", "decoupler_fine"],
        how="left"
    )
    .fillna({"n": 0, "fraction": 0})
)

fine_within_coarse_long.to_csv("tma6_tumour_decoupler_frac_fine_on_coarse.csv", index = False)

### similar fraction calculation but on total #cells

In [ ]:
df = tma6_tumour.obs[["PatientID", "decoupler", "decoupler_fine", "decoupler_coarse"]].copy()

# Ensure categories
for col in ["PatientID", "decoupler", "decoupler_fine", "decoupler_coarse"]:
    df[col] = df[col].astype("category")

all_patients = df["PatientID"].cat.categories
all_coarse   = df["decoupler_coarse"].cat.categories
all_broad    = df["decoupler"].cat.categories
all_fine     = df["decoupler_fine"].cat.categories


In [ ]:
## coarse on total cells

coarse_counts = (
    df.groupby(["PatientID", "decoupler_coarse"])
      .size().rename("n").reset_index()
)

patient_totals = (
    df.groupby("PatientID")
      .size().rename("denom").reset_index()
)

coarse_on_whole = (
    coarse_counts
    .merge(patient_totals, on="PatientID", how="left")
)
coarse_on_whole["fraction"] = coarse_on_whole["n"] / coarse_on_whole["denom"]

valid_coarse_whole = (
    pd.MultiIndex.from_product(
        [all_patients, all_coarse],
        names=["PatientID", "decoupler_coarse"]
    )
    .to_frame(index=False)
)

coarse_on_whole = (
    valid_coarse_whole
    .merge(coarse_on_whole, on=["PatientID", "decoupler_coarse"], how="left")
    .fillna({"n": 0, "fraction": 0})
)

coarse_on_whole.to_csv("tma6_tumour_decoupler_frac_coarse_on_whole.csv", index = False)

In [ ]:
broad_counts = (
    df.groupby(["PatientID", "decoupler"])
      .size().rename("n").reset_index()
)

broad_on_whole = (
    broad_counts
    .merge(patient_totals, on="PatientID", how="left")
)
broad_on_whole["fraction"] = broad_on_whole["n"] / broad_on_whole["denom"]

valid_broad_whole = (
    pd.MultiIndex.from_product(
        [all_patients, all_broad],
        names=["PatientID", "decoupler"]
    )
    .to_frame(index=False)
)

broad_on_whole = (
    valid_broad_whole
    .merge(broad_on_whole, on=["PatientID", "decoupler"], how="left")
    .fillna({"n": 0, "fraction": 0})
)

broad_on_whole.to_csv("tma6_tumour_decoupler_frac_broad_on_whole.csv", index = False)

In [ ]:
fine_mask = df["decoupler_fine"].astype(str) != df["decoupler"].astype(str)
fine_df = df[fine_mask]

fine_counts = (
    fine_df.groupby(["PatientID", "decoupler_fine"])
           .size().rename("n").reset_index()
)

fine_on_whole = (
    fine_counts
    .merge(patient_totals, on="PatientID", how="left")
)
fine_on_whole["fraction"] = fine_on_whole["n"] / fine_on_whole["denom"]


true_fine_labels = fine_df["decoupler_fine"].unique()

valid_fine_whole = (
    pd.MultiIndex.from_product(
        [all_patients, true_fine_labels],
        names=["PatientID", "decoupler_fine"]
    )
    .to_frame(index=False)
)

fine_on_whole = (
    valid_fine_whole
    .merge(fine_on_whole, on=["PatientID", "decoupler_fine"], how="left")
    .fillna({"n": 0, "fraction": 0})
)

fine_on_whole.to_csv("tma6_tumour_decoupler_frac_fine_on_whole.csv", index = False)

## pseudobulk of the coarse, broad and fine labels

In [ ]:
adata = tma6_tumour.copy()

In [ ]:
adata.X = adata.layers['raw_counts'].copy()
adata.X[1:5, 1:5].toarray()

In [ ]:
adata

### pseudobulk ignoring #of cells (bulk-normalisation before survival)

In [ ]:
obs_columns = ['decoupler_coarse', 'PatientID']

counts_df = pd.DataFrame(
    adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X,
    index=adata.obs_names,
    columns=adata.var_names
)

meta = adata.obs[obs_columns]
combined_df = pd.concat([meta, counts_df], axis=1)

# Group by the desired labels and sum
pseudobulk_df = combined_df.groupby(obs_columns).sum()

pb_long = (
    pseudobulk_df
    .reset_index()
    .melt(
        id_vars=['decoupler_coarse', 'PatientID'],
        var_name='symbol',
        value_name='count'
    )
    .rename(columns={'decoupler_coarse': 'cell_type',
                     'PatientID': 'sample'})
)

pb_long.to_csv("tma6_tumour_decoupler_coarse_summed_bulk.csv", index = False)

In [ ]:
obs_columns = ['decoupler', 'PatientID']

counts_df = pd.DataFrame(
    adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X,
    index=adata.obs_names,
    columns=adata.var_names
)

meta = adata.obs[obs_columns]
combined_df = pd.concat([meta, counts_df], axis=1)

# Group by the desired labels and sum
pseudobulk_df = combined_df.groupby(obs_columns).sum()

pb_long = (
    pseudobulk_df
    .reset_index()
    .melt(
        id_vars=['decoupler', 'PatientID'],
        var_name='symbol',
        value_name='count'
    )
    .rename(columns={'decoupler': 'cell_type',
                     'PatientID': 'sample'})
)

pb_long.to_csv("tma6_tumour_decoupler_broad_summed_bulk.csv", index = False)

In [ ]:
fine_mask = adata.obs["decoupler_fine"].astype(str) != adata.obs["decoupler"].astype(str)
adata_fine = adata[fine_mask].copy()

obs_columns = ['decoupler_fine', 'PatientID']

counts_df = pd.DataFrame(
    adata_fine.X.toarray() if hasattr(adata_fine.X, 'toarray') else adata_fine.X,
    index=adata_fine.obs_names,
    columns=adata_fine.var_names
)

meta = adata_fine.obs[obs_columns]
combined_df = pd.concat([meta, counts_df], axis=1)

# Group by PatientID × fine label
pseudobulk_df = combined_df.groupby(obs_columns).sum()

pb_long = (
    pseudobulk_df
    .reset_index()
    .melt(
        id_vars=['decoupler_fine', 'PatientID'],
        var_name='symbol',
        value_name='count'
    )
    .rename(columns={
        'decoupler_fine': 'cell_type',
        'PatientID': 'sample'
    })
)

pb_long.to_csv("tma6_tumour_decoupler_fine_summed_bulk.csv", index = False)

### pseudobulk averaged expression #of cells

In [ ]:
# coarse labels
obs_columns = ['decoupler_coarse', 'PatientID']

counts_df = pd.DataFrame(
    adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X,
    index=adata.obs_names,
    columns=adata.var_names
)

meta = adata.obs[obs_columns]
combined_df = pd.concat([meta, counts_df], axis=1)

# Summed pseudobulk
pseudobulk_df = combined_df.groupby(obs_columns).sum()

cell_counts = combined_df.groupby(obs_columns).size()

pseudobulk_mean_df = pseudobulk_df.div(cell_counts, axis=0)
pb_long = (
    pseudobulk_mean_df
    .reset_index()
    .melt(
        id_vars=['decoupler_coarse', 'PatientID'],
        var_name='symbol',
        value_name='count'
    )
    .rename(columns={
        'decoupler_coarse': 'cell_type',
        'PatientID': 'sample'
    })
)

pb_long.to_csv("tma6_tumour_decoupler_coarse_mean_bulk.csv", index=False)


In [ ]:
# coarse labels
obs_columns = ['decoupler', 'PatientID']

counts_df = pd.DataFrame(
    adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X,
    index=adata.obs_names,
    columns=adata.var_names
)

meta = adata.obs[obs_columns]
combined_df = pd.concat([meta, counts_df], axis=1)

# Summed pseudobulk
pseudobulk_df = combined_df.groupby(obs_columns).sum()

cell_counts = combined_df.groupby(obs_columns).size()

pseudobulk_mean_df = pseudobulk_df.div(cell_counts, axis=0)
pb_long = (
    pseudobulk_mean_df
    .reset_index()
    .melt(
        id_vars=['decoupler', 'PatientID'],
        var_name='symbol',
        value_name='count'
    )
    .rename(columns={
        'decoupler': 'cell_type',
        'PatientID': 'sample'
    })
)

pb_long.to_csv("tma6_tumour_decoupler_broad_mean_bulk.csv", index=False)


In [ ]:
fine_mask = adata.obs["decoupler_fine"].astype(str) != adata.obs["decoupler"].astype(str)
adata_fine = adata[fine_mask].copy()


obs_columns = ['decoupler_fine', 'PatientID']


counts_df = pd.DataFrame(
    adata_fine.X.toarray() if hasattr(adata_fine.X, 'toarray') else adata_fine.X,
    index=adata_fine.obs_names,
    columns=adata_fine.var_names
)


meta = adata_fine.obs[obs_columns]


combined_df = pd.concat([meta, counts_df], axis=1)


pseudobulk_df = combined_df.groupby(obs_columns).sum()
cell_counts = combined_df.groupby(obs_columns).size()


pseudobulk_mean_df = pseudobulk_df.div(cell_counts, axis=0)


pb_long = (
    pseudobulk_mean_df
    .reset_index()
    .melt(
        id_vars=['decoupler_fine', 'PatientID'],
        var_name='symbol',
        value_name='count'
    )
    .rename(columns={
        'decoupler_fine': 'cell_type',
        'PatientID': 'sample'
    })
)

pb_long.to_csv("tma6_tumour_decoupler_fine_mean_bulk.csv", index=False)


### counting gene+ve cells per sample

In [ ]:
obs_columns = ['decoupler_coarse', 'PatientID']

counts_df = pd.DataFrame(
    adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X,
    index=adata.obs_names,
    columns=adata.var_names
)

meta = adata.obs[obs_columns]
combined_df = pd.concat([meta, counts_df], axis=1)

binary_df = (combined_df[adata.var_names] > 0).astype(int)

binary_df = pd.concat([meta, binary_df], axis=1)

fraction_df = (
    binary_df
    .groupby(obs_columns)
    .mean()   
)

fraction_long = (
    fraction_df
    .reset_index()
    .melt(
        id_vars=['decoupler_coarse', 'PatientID'],
        var_name='symbol',
        value_name='fraction_positive'
    )
    .rename(columns={
        'decoupler_coarse': 'cell_type',
        'PatientID': 'sample'
    })
)

fraction_long.to_csv(
    "tma6_tumour_decoupler_coarse_fraction_positive.csv",
    index=False
)

In [ ]:
obs_columns = ['decoupler', 'PatientID']

counts_df = pd.DataFrame(
    adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X,
    index=adata.obs_names,
    columns=adata.var_names
)

meta = adata.obs[obs_columns]
combined_df = pd.concat([meta, counts_df], axis=1)

binary_df = (combined_df[adata.var_names] > 0).astype(int)

binary_df = pd.concat([meta, binary_df], axis=1)

fraction_df = (
    binary_df
    .groupby(obs_columns)
    .mean()   
)

fraction_long = (
    fraction_df
    .reset_index()
    .melt(
        id_vars=['decoupler', 'PatientID'],
        var_name='symbol',
        value_name='fraction_positive'
    )
    .rename(columns={
        'decoupler': 'cell_type',
        'PatientID': 'sample'
    })
)

fraction_long.to_csv(
    "tma6_tumour_decoupler_broad_fraction_positive.csv",
    index=False
)

In [ ]:
fine_mask = adata.obs["decoupler_fine"].astype(str) != adata.obs["decoupler"].astype(str)
adata_fine = adata[fine_mask].copy()

obs_columns = ['decoupler_fine', 'PatientID']

counts_df = pd.DataFrame(
    adata_fine.X.toarray() if hasattr(adata_fine.X, 'toarray') else adata_fine.X,
    index=adata_fine.obs_names,
    columns=adata_fine.var_names
)

meta = adata_fine.obs[obs_columns]
combined_df = pd.concat([meta, counts_df], axis=1)

binary_df = (combined_df[adata_fine.var_names] > 0).astype(int)

binary_df = pd.concat([meta, binary_df], axis=1)

fraction_df = (
    binary_df
    .groupby(obs_columns)
    .mean()   
)

fraction_long = (
    fraction_df
    .reset_index()
    .melt(
        id_vars=['decoupler_fine', 'PatientID'],
        var_name='symbol',
        value_name='fraction_positive'
    )
    .rename(columns={
        'decoupler_fine': 'cell_type',
        'PatientID': 'sample'
    })
)

fraction_long.to_csv(
    "tma6_tumour_decoupler_fine_fraction_positive.csv",
    index=False
)

In [ ]:
fraction_long.head()

### sanity check

In [ ]:
tma6_tumour

#### dotplots for sig HR genes in coarse Tumour bins

In [ ]:
sc.settings.figdir = "/homevol/kk/analysis/OVA_TMA/TMA6_notebook/adjustments_mis_assigned_transcripts/supervised_annotation/v3/survival/HR_genes_pseudobulk_of_celltypes"


In [ ]:
## dotplot for HR sig genes in other bins
## sum + TMM

genes = pd.read_csv(
    "/homevol/kk/analysis/OVA_TMA/TMA6_notebook/adjustments_mis_assigned_transcripts/"
    "supervised_annotation/v3/survival/HR_genes_pseudobulk_of_celltypes/"
    "with_aggregated_counts_TMM_normalised/coarse_labels/"
    "HR_median_HL_tma6_tumour_decoupler_all_Tumour_summed_bulk.csv"
)

sig_genes = genes.query("P < 0.05")["symbol"].tolist()

In [ ]:

sc.pl.dotplot(
    tma6_tumour,
    var_names=sig_genes,
    groupby="decoupler_coarse",
    figsize=(12, 2.5),   
    show=True,
    save = "dotplot_tma6_tumour_cores_coarse_tumour_sig_HR_genes_sum+TMM_exp.pdf"
)


In [ ]:
## dotplot for HR sig genes in other bins
## mean_counts

genes = pd.read_csv(
    "/homevol/kk/analysis/OVA_TMA/TMA6_notebook/adjustments_mis_assigned_transcripts/"
    "supervised_annotation/v3/survival/HR_genes_pseudobulk_of_celltypes/"
    "with_averaged_counts_for_cell_numbers/coarse_labels/"
    "HR_median_HL_tma6_tumour_decoupler_all_Tumour_mean_bulk.csv"
)

sig_genes = genes.query("P < 0.05")["symbol"].tolist()

In [ ]:

sc.pl.dotplot(
    tma6_tumour,
    var_names=sig_genes,
    groupby="decoupler_coarse",
    figsize=(12, 2.5),   
    show=True,
    save = "dotplot_tma6_tumour_cores_coarse_tumour_sig_HR_genes_raw_mean_exp.pdf"
)


In [ ]:
## dotplot for HR sig genes in other bins
## +gene_cell_frac

genes = pd.read_csv(
    "/homevol/kk/analysis/OVA_TMA/TMA6_notebook/adjustments_mis_assigned_transcripts/"
    "supervised_annotation/v3/survival/HR_genes_pseudobulk_of_celltypes/"
    "with_gene+_fractions/coarse_labels/"
    "HR_median_HL_tma6_tumour_decoupler_Tumour_gene+_frac.csv"
)

sig_genes = genes.query("P < 0.05")["symbol"].tolist()

In [ ]:

sc.pl.dotplot(
    tma6_tumour,
    var_names=sig_genes,
    groupby="decoupler_coarse",
    figsize=(12, 2.5),   
    show=True,
    save = "dotplot_tma6_tumour_cores_coarse_tumour_sig_HR_genes_+gene_cell_frac.pdf"
)


In [ ]:
sc.pl.dotplot(
    tma6_tumour,
    var_names=sig_genes,
    groupby="decoupler_coarse",
    figsize=(16, 2.5),   
    show=True,
    save = "dotplot_tma6_tumour_cores_coarse_Immune_sig_HR_genes_exp.pdf"
)


In [ ]:
### for mean exp

In [ ]:
## dotplot for HR sig genes in other bins

## for tumour

genes = pd.read_csv(
    "/homevol/kk/analysis/OVA_TMA/TMA6_notebook/adjustments_mis_assigned_transcripts/"
    "supervised_annotation/v3/survival/HR_genes_pseudobulk_of_celltypes/"
    "with_averaged_counts_for_cell_numbers/coarse_labels/"
    "HR_median_HL_tma6_tumour_decoupler_all_Tumour_mean_bulk.csv"
)

sig_genes = genes.query("P < 0.05")["symbol"].tolist()

In [ ]:
sc.pl.dotplot(
    tma6_tumour,
    var_names=sig_genes,
    groupby="decoupler_coarse",
    figsize=(12, 2.5),   
    show=True,
    save = "tma6_tumour_cores_coarse_tumour_sig_HR_genes_mean_exp.pdf"
)


In [ ]:
## dotplot for HR sig genes in other bins

## for stromal

genes = pd.read_csv(
    "/homevol/kk/analysis/OVA_TMA/TMA6_notebook/adjustments_mis_assigned_transcripts/"
    "supervised_annotation/v3/survival/HR_genes_pseudobulk_of_celltypes/"
    "with_averaged_counts_for_cell_numbers/coarse_labels/"
    "HR_median_HL_tma6_tumour_decoupler_all_Stromal_mean_bulk.csv"
)

sig_genes = genes.query("P < 0.05")["symbol"].tolist()

In [ ]:
sc.pl.dotplot(
    tma6_tumour,
    var_names=sig_genes,
    groupby="decoupler_coarse",
    figsize=(12, 2.5),   
    show=True,
    save = "tma6_tumour_cores_coarse_stromal_sig_HR_genes_mean_exp.pdf"
)


In [ ]:
## dotplot for HR sig genes in other bins

## for immune

genes = pd.read_csv(
    "/homevol/kk/analysis/OVA_TMA/TMA6_notebook/adjustments_mis_assigned_transcripts/"
    "supervised_annotation/v3/survival/HR_genes_pseudobulk_of_celltypes/"
    "with_averaged_counts_for_cell_numbers/coarse_labels/"
    "HR_median_HL_tma6_tumour_decoupler_all_Immune_mean_bulk.csv"
)

sig_genes = genes.query("P < 0.05")["symbol"].tolist()

In [ ]:
sc.pl.dotplot(
    tma6_tumour,
    var_names=sig_genes,
    groupby="decoupler_coarse",
    figsize=(16, 2.5),   
    show=True,
    save = "tma6_tumour_cores_coarse_Immune_sig_HR_genes_mean_exp.pdf"
)


In [ ]:
## dotplot for HR sig genes in other bins

## for immune

genes = pd.read_csv(
    "/homevol/kk/analysis/OVA_TMA/TMA6_notebook/adjustments_mis_assigned_transcripts/"
    "supervised_annotation/v3/survival/HR_genes_pseudobulk_of_celltypes/"
    "with_averaged_counts_for_cell_numbers/coarse/"
    "HR_median_HL_tma6_tumour_decoupler_all_Immune_mean_bulk.csv"
)

sig_genes = genes.query("P < 0.05")["symbol"].tolist()

In [ ]:
sc.pl.dotplot(
    tma6_tumour,
    var_names=sig_genes,
    groupby="decoupler_coarse",
    figsize=(16, 2.5),   
    show=True,
    save = "tma6_tumour_cores_coarse_Immune_sig_HR_genes_mean_exp.pdf"
)


### Adjust the TMA coordinates

In [ ]:
def map_coarsetype(x):
    if x in tumour_labels:
        return "Tumour"
    elif x in immune_labels:
        return "Immune"
    elif x in stromal_labels:
        return "Stromal"
    else:
        return "Other"

tma6.obs["decoupler_coarse"] = tma6.obs["decoupler"].map(map_coarsetype)


In [ ]:
sc.pl.spatial(
     tma6,
     color="decoupler_coarse",
     spot_size=20,      
     cmap="viridis"
 )

In [ ]:
adata = tma6.copy()

import numpy as np

# --- config ---
source_L = 'TMA6L'
source_R = 'TMA6R'
margin_y = 0.0  # initial gap reduction for whole TMA6R
shared_margin_y = 0.0  # final margin for shared cores

adata.obsm.setdefault('spatial_original', adata.obsm['spatial'].copy())
sp_new = adata.obsm['spatial'].copy()

# masks
mask_L_all = adata.obs['source'] == source_L
mask_R_all = adata.obs['source'] == source_R

# --- STEP 1: bring down whole TMA6R closer to L ---
y_L_max = np.max(sp_new[mask_L_all,1])  # bottom of all L
y_R_min = np.min(sp_new[mask_R_all,1])  # top of all R
delta_y_R = y_R_min - (y_L_max + margin_y)  # how much to move R down
sp_new[mask_R_all,1] -= delta_y_R

# provenance
adata.uns.setdefault('spatial_shifts', {})
adata.uns['spatial_shifts']['TMA6R_initial_shift'] = {
    'delta_y': float(delta_y_R),
    'method': 'bring_down_whole_TMA6R'
}

# --- STEP 2: align x and y for shared arrayIDs ---
ids_L = adata.obs.loc[mask_L_all, 'arrayID'].unique()
ids_R = adata.obs.loc[mask_R_all, 'arrayID'].unique()
shared_ids = np.intersect1d(ids_L, ids_R)

for array_id in shared_ids:
    mask_L = mask_L_all & (adata.obs['arrayID'] == array_id)
    mask_R = mask_R_all & (adata.obs['arrayID'] == array_id)
    
    if mask_L.sum()==0 or mask_R.sum()==0:
        continue
    
    # --- horizontal alignment ---
    x_L_center = np.median(sp_new[mask_L,0])
    x_R_center = np.median(sp_new[mask_R,0])
    delta_x = x_R_center - x_L_center
    sp_new[mask_L,0] += delta_x
    
    # --- vertical adjustment for shared core ---
    y_L_max_core = np.max(sp_new[mask_L,1])
    y_R_min_core = np.min(sp_new[mask_R,1])
    delta_y = y_L_max_core - y_R_min_core - shared_margin_y
    sp_new[mask_L,1] -= delta_y
    
    # provenance per core
    adata.uns['spatial_shifts'][f'{array_id}_shared_snap'] = {
        'delta_x': float(delta_x),
        'delta_y': float(delta_y),
        'margin_y': shared_margin_y,
        'method': 'shared_core_xy_snap_after_global_R'
    }

adata.obsm['spatial'] = sp_new
print(f"TMA6R moved down by {delta_y_R:.2f} µm. Shared cores aligned in x and y with margin {shared_margin_y} µm.")

In [ ]:
delta_y

In [ ]:
obs = adata.obs
ids_L = obs.loc[obs['source'] == source_L, 'arrayID'].unique()
ids_R = obs.loc[obs['source'] == source_R, 'arrayID'].unique()
shared_ids = np.intersect1d(ids_L, ids_R)

sc.pl.spatial(adata[adata.obs['arrayID'].isin(shared_ids)], color='source', spot_size=20, cmap='viridis')

In [ ]:
sc.pl.spatial(adata, color='source', spot_size=20, cmap='viridis')

In [ ]:
adata

In [ ]:
adata.write_h5ad("/homevol/kk/analysis/OVA_TMA/TMA6_notebook/adjustments_mis_assigned_transcripts/supervised_annotation/v3/tma6_xen_mist_coarse_decoupler_filt_fine_sp_coord_adj.h5ad")

## with resolvi

In [ ]:
res_tma6 = sc.read("/homevol/kk/analysis/OVA_TMA/TMA6_notebook/adjustments_mis_assigned_transcripts/supervised_annotation/v3/Resolvi_test/resolvi_tma6_xen_mist_coarse_decoupler_filt.h5ad")

In [ ]:
res_tma6

In [ ]:
dec = res_tma6.obs["decoupler"].astype(str)
resv = res_tma6.obs["resolvi_predicted"].astype(str)
cross = pd.crosstab(dec, resv)
cross_norm = cross.div(cross.sum(axis=1), axis=0)
cross_norm_col = cross.div(cross.sum(axis=0), axis=1)
agreement = pd.DataFrame({
    "decoupler_label": cross.index,
    "top_resolvi_label": cross.idxmax(axis=1),
    "agreement_fraction": cross.max(axis=1) / cross.sum(axis=1)
})
agreement.sort_values("agreement_fraction", ascending=False)


In [ ]:
overall_agreement = (dec == resv).mean()
overall_agreement


In [ ]:
plt.figure(figsize=(12,10))
sns.heatmap(cross_norm, cmap="Blues", annot=False)
plt.title("Decoupler vs ResolVI — Normalized Confusion Matrix")
plt.show()


In [ ]:
sc.pp.neighbors(res_tma6, use_rep="X_resolVI")
sc.tl.umap(res_tma6)

In [ ]:
with plt.rc_context({'figure.figsize': (10, 8), 'font.size': 16}):
    sc.pl.umap(res_tma6, color="resolvi_predicted", size = 0.6)


In [ ]:
with plt.rc_context({'figure.figsize': (10, 8), 'font.size': 16}):
    sc.pl.umap(
        res_tma6[res_tma6.obs["Diagnosis"] == "Tumour"].copy(),
        color="resolvi_predicted",
        size=0.6
    )


In [ ]:
res_tma6.obs["PatientID"].nunique()

## with annotatability

In [ ]:
annot_tma6 = sc.read("/homevol/kk/analysis/OVA_TMA/TMA6_notebook/adjustments_mis_assigned_transcripts/supervised_annotation/v3/Annotatability_test/annotatability_tma6_xen_mist_coarse_decoupler_filt.h5ad")

In [ ]:
annot_tma6

In [ ]:
dec = annot_tma6.obs["decoupler"].astype(str)
anno = annot_tma6.obs["CorrectedCellType"].astype(str)

# Confusion matrix
cross = pd.crosstab(dec, anno)

# Row-normalized (per decoupler label)
cross_norm = cross.div(cross.sum(axis=1), axis=0)

# Column-normalized (per Annotatability label)
cross_norm_col = cross.div(cross.sum(axis=0), axis=1)

# Agreement table
agreement = pd.DataFrame({
    "decoupler_label": cross.index,
    "top_annot_label": cross.idxmax(axis=1),
    "agreement_fraction": cross.max(axis=1) / cross.sum(axis=1)
})

# Sort by agreement_fraction (highest first)
agreement_sorted = agreement.sort_values("agreement_fraction", ascending=False)
agreement_sorted
